# 22.08 - Mask R-CNN OOF evaluation and fold ensembling

**Notebook type:** Solution notebook with completed exercises, smoke checks, and test cases.

**Daily output:** Complete OOF instance metrics, threshold evidence, a class-aware fold ensemble, and submission-style RLE.

## Core ideas

- Tune thresholds only on complete out-of-fold predictions: each training image is predicted by the fold model that never saw it.
- Match predictions to ground truth within the same image and class. Sort by confidence and allow each ground-truth instance to match once; duplicates are false positives.
- This notebook reports mask-IoU-based macro-F1 for fast local decisions. A final submission must use the competition's official evaluator (often COCO-style AP over several IoUs).
- Fold ensembling is not plain pixel averaging: first associate same-class instances, then combine their mask probabilities.
- RLE order varies by competition. The helper below uses common column-major, one-indexed start/length runs; verify the exact rules before submission.

In [ ]:
import itertools

import numpy as np
import pandas as pd
import torch
from torchvision.ops import masks_to_boxes

SEED = 22
np.random.seed(SEED)
torch.manual_seed(SEED)
HEIGHT = WIDTH = 32
NUM_CLASSES = 3


## Prepared OOF records

The records below mimic raw Mask R-CNN validation caches from three folds. Every stable image ID occurs once, masks remain soft until evaluation, and deliberate low-confidence duplicates make threshold selection meaningful. No training occurs here because evaluation should be cheap and repeatable.

In [ ]:
OOF_RECORDS = []
for image_id in range(12):
    class_id = 1 + image_id % 2
    fold = image_id % 3
    top = 4 + image_id % 4
    left = 5 + (image_id * 2) % 5
    gt_mask = torch.zeros((1, HEIGHT, WIDTH), dtype=torch.uint8)
    gt_mask[0, top:top + 13, left:left + 12] = 1
    good_probability = gt_mask[0].to(torch.float32) * 0.82 + 0.08
    duplicate_probability = torch.roll(good_probability, shifts=2, dims=1) * 0.72
    false_probability = torch.zeros((HEIGHT, WIDTH), dtype=torch.float32)
    false_probability[20:27, 20:28] = 0.68
    OOF_RECORDS.append(
        {
            "image_id": image_id,
            "fold": fold,
            "gt_labels": torch.tensor([class_id], dtype=torch.int64),
            "gt_masks": gt_mask,
            "labels": torch.tensor([class_id, class_id, 3 - class_id], dtype=torch.int64),
            "scores": torch.tensor([0.88 - 0.02 * (image_id % 3), 0.38, 0.22], dtype=torch.float32),
            "masks": torch.stack([good_probability, duplicate_probability, false_probability])[:, None],
        }
    )

print(pd.DataFrame({"image_id": [r["image_id"] for r in OOF_RECORDS], "fold": [r["fold"] for r in OOF_RECORDS]}).groupby("fold").size())


## Exercise 22-A: Vectorized pairwise mask IoU

Flatten spatial dimensions, broadcast the instance axes, and calculate intersection over union. Empty sets must return a correctly shaped empty matrix.

**Return structure — `pairwise_mask_iou`:** A CPU `torch.float32` tensor `[P,G]`, where `P` is the number of predicted masks and `G` the number of target masks. Each value lies in `[0,1]`. Inputs are binary-like tensors `[P,H,W]` and `[G,H,W]`.

In [ ]:
def pairwise_mask_iou(prediction_masks, target_masks):
    prediction_masks = prediction_masks.to(torch.bool).flatten(1)
    target_masks = target_masks.to(torch.bool).flatten(1)
    if prediction_masks.shape[0] == 0 or target_masks.shape[0] == 0:
        return torch.zeros((prediction_masks.shape[0], target_masks.shape[0]), dtype=torch.float32)
    intersection = (prediction_masks[:, None] & target_masks[None, :]).sum(dim=2).to(torch.float32)
    union = (prediction_masks[:, None] | target_masks[None, :]).sum(dim=2).to(torch.float32)
    return intersection / union.clamp_min(1.0)

# Smoke check: identical masks have IoU 1.
smoke_iou = pairwise_mask_iou(OOF_RECORDS[0]["gt_masks"], OOF_RECORDS[0]["gt_masks"])
print(smoke_iou)


## Exercise 22-B: Class-aware one-to-one matching

Apply score and mask thresholds, then greedily match score-sorted predictions to unmatched same-class targets at the chosen IoU.

**Return structure — `evaluate_instance_records`:** A `dict` with integer `images`, `predicted_instances`, and `ground_truth_instances`; float `macro_f1` and `mean_matched_iou`; and `per_class`, a list of `num_classes-1` dictionaries. Each class row has integer `class_id`, `tp`, `fp`, `fn` and float `precision`, `recall`, `f1`, `mean_matched_iou`.

In [ ]:
def evaluate_instance_records(records, score_threshold=0.5, mask_threshold=0.5, iou_threshold=0.5, num_classes=3):
    rows = []
    matched_ious_all = []
    total_predictions = 0
    total_ground_truth = 0
    for class_id in range(1, int(num_classes)):
        true_positives = 0
        false_positives = 0
        false_negatives = 0
        class_matched_ious = []
        for record in records:
            gt_selector = record["gt_labels"] == class_id
            gt_masks = record["gt_masks"][gt_selector].to(torch.bool)
            pred_selector = (record["labels"] == class_id) & (record["scores"] >= float(score_threshold))
            pred_scores = record["scores"][pred_selector]
            pred_masks = record["masks"][pred_selector]
            if pred_masks.ndim == 4:
                pred_masks = pred_masks[:, 0]
            pred_masks = pred_masks >= float(mask_threshold)
            order = torch.argsort(pred_scores, descending=True)
            pred_masks = pred_masks[order]
            ious = pairwise_mask_iou(pred_masks, gt_masks)
            unmatched = set(range(int(gt_masks.shape[0])))
            for prediction_index in range(int(pred_masks.shape[0])):
                candidates = [(float(ious[prediction_index, gt_index]), gt_index) for gt_index in unmatched]
                best_iou, best_index = max(candidates, default=(0.0, -1))
                if best_iou >= float(iou_threshold):
                    true_positives += 1
                    unmatched.remove(best_index)
                    class_matched_ious.append(best_iou)
                else:
                    false_positives += 1
            false_negatives += len(unmatched)
        precision = true_positives / max(true_positives + false_positives, 1)
        recall = true_positives / max(true_positives + false_negatives, 1)
        f1 = 2.0 * precision * recall / max(precision + recall, 1e-12)
        rows.append(
            {
                "class_id": class_id,
                "tp": true_positives,
                "fp": false_positives,
                "fn": false_negatives,
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "mean_matched_iou": float(np.mean(class_matched_ious)) if class_matched_ious else 0.0,
            }
        )
        matched_ious_all.extend(class_matched_ious)
        total_predictions += true_positives + false_positives
        total_ground_truth += true_positives + false_negatives
    return {
        "images": len(records),
        "predicted_instances": total_predictions,
        "ground_truth_instances": total_ground_truth,
        "macro_f1": float(np.mean([row["f1"] for row in rows])),
        "mean_matched_iou": float(np.mean(matched_ious_all)) if matched_ious_all else 0.0,
        "per_class": rows,
    }

# Smoke check and complete-OOF evidence.
oof_metrics = evaluate_instance_records(OOF_RECORDS, score_threshold=0.5, mask_threshold=0.5)
print({key: value for key, value in oof_metrics.items() if key != "per_class"})
print(pd.DataFrame(oof_metrics["per_class"]).round(3).to_string(index=False))


## Exercise 22-C: Tune cached OOF thresholds

Sweep cached predictions; do not rerun inference and do not select thresholds on test images.

**Return structure — `sweep_oof_thresholds`:** A `pandas.DataFrame` with one row per Cartesian threshold pair and columns `score_threshold`, `mask_threshold`, `macro_f1`, `mean_matched_iou`, `predicted_instances`, and `ground_truth_instances`, ranked by macro-F1 then matched IoU.

In [ ]:
def sweep_oof_thresholds(records, score_thresholds, mask_thresholds, iou_threshold=0.5, num_classes=3):
    rows = []
    for score_threshold, mask_threshold in itertools.product(score_thresholds, mask_thresholds):
        result = evaluate_instance_records(
            records,
            score_threshold=float(score_threshold),
            mask_threshold=float(mask_threshold),
            iou_threshold=float(iou_threshold),
            num_classes=int(num_classes),
        )
        rows.append(
            {
                "score_threshold": float(score_threshold),
                "mask_threshold": float(mask_threshold),
                "macro_f1": result["macro_f1"],
                "mean_matched_iou": result["mean_matched_iou"],
                "predicted_instances": result["predicted_instances"],
                "ground_truth_instances": result["ground_truth_instances"],
            }
        )
    return pd.DataFrame(rows).sort_values(
        ["macro_f1", "mean_matched_iou", "score_threshold"], ascending=[False, False, False]
    ).reset_index(drop=True)

# Smoke check: compare all threshold pairs on every OOF image.
threshold_table = sweep_oof_thresholds(OOF_RECORDS, score_thresholds=[0.2, 0.5, 0.8], mask_thresholds=[0.4, 0.5, 0.7])
best_thresholds = threshold_table.iloc[0]
print(threshold_table.round(3).to_string(index=False))


## Exercise 22-D: Associate and ensemble fold masks

For one test image, gather all fold outputs, discard low scores, cluster only same-class masks above `merge_iou`, and score-weight their soft masks. This compact rule is a practical baseline; crowded scenes may need stronger assignment logic.

**Return structure — `ensemble_fold_predictions`:** A Mask R-CNN-style dictionary with CPU `boxes` (`float32 [K,4]`), `labels` (`int64 [K]`), `scores` (`float32 [K]`), and soft `masks` (`float32 [K,1,H,W]`). `K` is the number of retained instance clusters and may be zero.

In [ ]:
def ensemble_fold_predictions(fold_predictions, score_threshold=0.25, mask_threshold=0.5, merge_iou=0.5):
    candidates = []
    for output in fold_predictions:
        masks = output["masks"][:, 0] if output["masks"].ndim == 4 else output["masks"]
        for mask, label, score in zip(masks, output["labels"], output["scores"]):
            if float(score) >= float(score_threshold):
                candidates.append({"mask": mask.to(torch.float32), "label": int(label), "score": float(score)})
    candidates.sort(key=lambda item: item["score"], reverse=True)
    clusters = []
    for candidate in candidates:
        best_cluster = -1
        best_iou = 0.0
        for cluster_index, cluster in enumerate(clusters):
            if cluster["label"] != candidate["label"]:
                continue
            iou = float(
                pairwise_mask_iou(
                    (candidate["mask"] >= float(mask_threshold))[None],
                    (cluster["mask"] >= float(mask_threshold))[None],
                )[0, 0]
            )
            if iou > best_iou:
                best_iou, best_cluster = iou, cluster_index
        if best_cluster >= 0 and best_iou >= float(merge_iou):
            cluster = clusters[best_cluster]
            new_weight = cluster["weight"] + candidate["score"]
            cluster["mask"] = (
                cluster["mask"] * cluster["weight"] + candidate["mask"] * candidate["score"]
            ) / new_weight
            cluster["weight"] = new_weight
            cluster["scores"].append(candidate["score"])
        else:
            clusters.append(
                {
                    "mask": candidate["mask"].clone(),
                    "label": candidate["label"],
                    "weight": candidate["score"],
                    "scores": [candidate["score"]],
                }
            )
    if not clusters:
        height, width = fold_predictions[0]["masks"].shape[-2:]
        return {
            "boxes": torch.zeros((0, 4), dtype=torch.float32),
            "labels": torch.zeros(0, dtype=torch.int64),
            "scores": torch.zeros(0, dtype=torch.float32),
            "masks": torch.zeros((0, 1, height, width), dtype=torch.float32),
        }
    probabilities = torch.stack([cluster["mask"] for cluster in clusters])
    labels = torch.tensor([cluster["label"] for cluster in clusters], dtype=torch.int64)
    scores = torch.tensor([float(np.mean(cluster["scores"])) for cluster in clusters], dtype=torch.float32)
    boxes = masks_to_boxes((probabilities >= float(mask_threshold)).to(torch.uint8))
    return {"boxes": boxes, "labels": labels, "scores": scores, "masks": probabilities[:, None]}

# Smoke check: two folds describe the same test object with slightly shifted probabilities.
base = OOF_RECORDS[0]
fold_predictions = [
    {"labels": base["labels"][:1], "scores": torch.tensor([0.86]), "masks": base["masks"][:1]},
    {"labels": base["labels"][:1], "scores": torch.tensor([0.82]), "masks": torch.roll(base["masks"][:1], shifts=1, dims=3)},
]
ensemble_output = ensemble_fold_predictions(fold_predictions, merge_iou=0.5)
print({key: tuple(value.shape) for key, value in ensemble_output.items()})


## Exercise 22-E: Encode submission masks

Threshold the selected ensemble probabilities and encode each instance. Keep `image_id`, class, score, and RLE together; audit row counts and empty-image rules against the competition specification.

**Return structure — `rle_encode`:** A Python `str` containing space-separated, one-indexed `start length` integer pairs for a 2D binary mask flattened in column-major order. An empty mask returns `""`.

In [ ]:
def rle_encode(mask):
    pixels = mask.to(torch.uint8).t().contiguous().reshape(-1).cpu().numpy()
    padded = np.concatenate([np.array([0], dtype=np.uint8), pixels, np.array([0], dtype=np.uint8)])
    changes = np.flatnonzero(padded[1:] != padded[:-1]) + 1
    changes[1::2] -= changes[::2]
    return " ".join(str(int(value)) for value in changes)

# Smoke check: encode the first ensembled instance.
submission_rle = rle_encode(ensemble_output["masks"][0, 0] >= float(best_thresholds["mask_threshold"]))
print("RLE prefix:", submission_rle[:80])


## Test Cases

Tests cover OOF uniqueness, pairwise IoU, complete evaluation counts, threshold-grid size, ensemble schema, and RLE round-trip reconstruction.

**Return structure — `run_day22_tests`:** Returns `None`; assertions communicate failure and `Day 22 tests passed` communicates success.

In [ ]:
def run_day22_tests():
    image_ids = [record["image_id"] for record in OOF_RECORDS]
    assert len(image_ids) == len(set(image_ids)) == 12
    assert sorted({record["fold"] for record in OOF_RECORDS}) == [0, 1, 2]
    assert smoke_iou.shape == (1, 1) and torch.allclose(smoke_iou, torch.ones(1, 1))
    empty_iou = pairwise_mask_iou(torch.zeros((0, 4, 4)), torch.zeros((2, 4, 4)))
    assert empty_iou.shape == (0, 2) and empty_iou.dtype == torch.float32
    assert oof_metrics["images"] == 12 and oof_metrics["ground_truth_instances"] == 12
    assert len(oof_metrics["per_class"]) == 2 and 0.0 <= oof_metrics["macro_f1"] <= 1.0
    assert len(threshold_table) == 9
    assert set(threshold_table.columns) == {"score_threshold", "mask_threshold", "macro_f1", "mean_matched_iou", "predicted_instances", "ground_truth_instances"}
    assert set(ensemble_output) == {"boxes", "labels", "scores", "masks"}
    count = int(ensemble_output["labels"].numel())
    assert ensemble_output["boxes"].shape == (count, 4) and ensemble_output["masks"].shape == (count, 1, HEIGHT, WIDTH)
    binary_mask = ensemble_output["masks"][0, 0] >= float(best_thresholds["mask_threshold"])
    values = [int(value) for value in submission_rle.split()]
    reconstructed_flat = np.zeros(HEIGHT * WIDTH, dtype=np.uint8)
    for start, length in zip(values[0::2], values[1::2]):
        reconstructed_flat[start - 1:start - 1 + length] = 1
    reconstructed = torch.from_numpy(reconstructed_flat.reshape(WIDTH, HEIGHT).T.copy()).to(torch.bool)
    assert torch.equal(reconstructed, binary_mask.cpu())
    print("Day 22 tests passed")

run_day22_tests()


## Day 22 Checklist

- [ ] Verify every training image has exactly one OOF prediction source.
- [ ] Match only within image and class, with one ground-truth instance used once.
- [ ] Tune score and mask thresholds on complete OOF records only.
- [ ] Report macro/per-class metrics and fold stability; use the official scorer for final decisions.
- [ ] Ensemble associated same-class instances, not unrelated pixels.
- [ ] Verify RLE order, indexing, empty-image behavior, IDs, and row count against the competition rules.
- [ ] Run the test cases.